# 02. Dựng bài toán và đóng băng nó

Chạy toàn bộ pipeline, chọn $\lambda$ bằng cross-validation 5 fold với quy tắc một sai
số chuẩn, tính $L$, $\mu$, $\kappa$, $f^*$ cho hai quy mô.

**Sau notebook này, `data/processed/` không được sửa nữa.** Mọi so sánh giữa các thuật
toán chỉ có nghĩa khi chúng cùng làm việc trên một hàm mục tiêu.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import prepare
report = prepare("../data/raw/accepted_2007_to_2018Q4.csv.gz", out_dir="../data/processed")

read 2,260,701 rows x 54 columns in 0s


after cleaning: 2,260,668 rows


categorical levels: 70 one-hot columns


lambda: CV minimum at lambda=1.778e-03 with MSE 12.042977; one-SE rule picks lambda=3.162e-02 with MSE 12.060672  (0s)
 sweep: n=200,000 d=116 L=9.1156 mu=0.034073 kappa=267.5 f*=6.151114 rmse=3.4563


  full: n=1,200,000 d=116 L=9.0620 mu=0.034036 kappa=266.2 f*=6.142806 rmse=3.4646


wrote ../data/processed/problem_config.json


In [3]:
pd.DataFrame(report.scales).T[["n", "d", "L", "mu", "kappa", "f_star", "rmse_test"]]

,n,d,L,mu,kappa,f_star,rmse_test
sweep,200000.0,116.0,9.115609,0.034073,267.529709,6.151114,3.456312
full,1200000.0,116.0,9.062028,0.034036,266.249186,6.142806,3.464642


## Phổ trị riêng và sàn do $\lambda$ đặt

Trị riêng nhỏ nhất quyết định $\mu$ có do dữ liệu hay do $\lambda$ chi phối.

In [4]:
from src.dataset import load_processed, SWEEP
from src.figures import spectrum_figure, save_figure
obj, X_test, y_test, cfg = load_processed("../data/processed", SWEEP)
gram = obj.hessian.copy(); gram.flat[::obj.d + 1] -= obj.lam
ev = np.linalg.eigvalsh(gram)
print(f"lambda_min = {ev[0]:.3e},  lambda = {obj.lam:.5g},  mu = {obj.mu:.5f}")
print(f"lambda dong gop {obj.lam/obj.mu*100:.0f}% vao mu")
save_figure(spectrum_figure(ev, obj.lam), "spectrum", "../results/figures")

lambda_min = 2.450e-03,  lambda = 0.031623,  mu = 0.03407
lambda dong gop 93% vao mu


[PosixPath('../results/figures/spectrum.pdf'),
 PosixPath('../results/figures/spectrum.png')]

## Ảnh hưởng của cách chuẩn hóa cột

In [5]:
from src.dataset import build_scaling_variants
from src.experiment import run_scaling_comparison
variants = build_scaling_variants("../data/raw/accepted_2007_to_2018Q4.csv.gz")
_, table = run_scaling_comparison(variants, lam=obj.lam, out_dir="../results/raw")
pd.DataFrame(table)

scaling-kappa: loaded 3 runs from ../results/raw/scaling-kappa.json


,scaling,L,mu,kappa,iters_to_1e-6,final_gap,status,rmse_test
0,raw,1.224746e+11,0.031628,3.872376e+12,NaN,3.743414e+00,max_iter,3.510286
1,center,6.038545e+10,0.031628,1.909255e+12,NaN,3.845810e+00,max_iter,3.479654
2,standardize,9.115609e+00,0.034073,2.675297e+02,630.0,3.520998e-18,converged,3.463367
